# EDA - Swedavia FlightInfo (Sandbox data)

**Purpose:** Answer the questions about the dimensional model depending on: `Grain`, `Identity`, `Status`, `Delay`, `Codeshare`, `Date` with `measured` numbers.

**Source:** `data/sandbox_swedavia_merge.duckdb`, schema: `swedavia_raw`, loaded by `dlt` with `merge disposition` (Arrivals and departures).


In [1]:
# Connections cell
from pathlib import Path
import duckdb

# Notebook bor i notebooks/. En nivå upp är repo root
REPO_ROOT = Path.cwd().parent
DB_PATH = REPO_ROOT / "data" / "sandbox_swedavia_merge.duckdb"
assert DB_PATH.exists(), f"Did not find {DB_PATH}"

# read_only för att jag behöver bara läsa här än så länge, DuckDB tillåter endast EN writing process
con = duckdb.connect(str(DB_PATH), read_only=True)

## 1) The shape of the data

A) The tables size and timespan. How many rows, and what time window do the scheduled times cover?

B) Swedaviass airport field varies by direction:
- arrival -> arrival_airport, departure -> departure_airport
- AT TIME ZONE converts UTC to Swedish time BEFORE extracting the date

---

A)

| source_table | n_rows | first_scheduled | last_scheduled |
|:--|:--|:--|:--|
| arrivals| 2649 | 2026-10-05 00:00:00+02:00 | 2026-10-09 23:55:00+02:00 |
| departures | 2635 | 2026-10-05 00:00:00+02:00 | 2026-10-09 23:25:00+02:00 |

---

B) 

| direction	| swedavia_airport | 2026-10-05 | 2026-10-06 | 2026-10-07 | 2026-10-08 | 
|:--|:--|:--:|:--|:--|:--|
| arrival | ARN | 362 | 351 | 349 | 352 | 
| arrival | BMA | 20 | 23 | 23 | 22 | 
| arrival | GOT | 80 | 69 | 77 | 82 | 
| arrival | KRN | 2	| 2 | 2	| 2 | 
| arrival | LLA | 15 | 	13 | 15 | 15 | 
| arrival | MMX | 23 | 	21 | 22 | 23 | 
| arrival | OSD | 6 | 6 | 6 | 6 | 
| arrival | RNB | 6 | 6 | 6 | 6 | 
| arrival | UME | 15 | 	14 | 15 | 14 | 
| arrival | VBY | 13 | 	12 | 12 | 13 | 
| departure | ARN | 355 | 352 | 346 | 349 | 
| departure | BMA | 20 | 24 | 23 | 21 | 
| departure | GOT | 79 | 72 | 76 | 72 | 
| departure | KRN | 2 | 2 | 2 | 2 | 
| departure | LLA | 15 | 13 | 15 | 17 | 
| departure | MMX | 22 | 20 | 22 | 23 | 
| departure | OSD | 6 | 7 | 7 | 6 | 
| departure | RNB | 6 | 6 | 6 | 5 | 
| departure | UME | 14 | 14 | 16 | 14 | 
| departure | VBY | 13 | 12 | 12 | 13 | 



In [2]:
# Hur många rader och timespan räcker tiderna
con.sql("""
    SELECT 'arrivals' AS source_table,
            COUNT(*) AS n_rows,
            MIN(arrival_time__scheduled_utc) AS first_scheduled,
            MAX(arrival_time__scheduled_utc) AS last_scheduled
    FROM swedavia_raw.arrivals
    UNION ALL
    SELECT 'departures',
            COUNT(*),
            MIN(departure_time__scheduled_utc),
            MAX(departure_time__scheduled_utc)
    FROM swedavia_raw.departures
""").df()

,source_table,n_rows,first_scheduled,last_scheduled
0,arrivals,2649,2026-10-05 00:00:00+02:00,2026-10-09 23:55:00+02:00
1,departures,2635,2026-10-05 00:00:00+02:00,2026-10-09 23:25:00+02:00


In [3]:
# Swedavias flygplats står på olika sidor.
# PIVOT statement gör att varje datum blir en egen kolumn
con.sql("""
    WITH movements AS (
        SELECT 'arrival' AS direction,
               flight_leg_identifier__arrival_airport_iata AS swedavia_airport,
               CAST(arrival_time__scheduled_utc AT TIME ZONE 'Europe/Stockholm' AS DATE) AS local_date
        FROM swedavia_raw.arrivals
        UNION ALL
        SELECT 'departure',
               flight_leg_identifier__departure_airport_iata,
               CAST(departure_time__scheduled_utc AT TIME ZONE 'Europe/Stockholm' AS DATE)
        FROM swedavia_raw.departures
    )
    PIVOT movements ON local_date USING COUNT(*)
    GROUP BY direction, swedavia_airport
    ORDER BY direction, swedavia_airport
""").df()

,direction,swedavia_airport,2026-10-05,2026-10-06,2026-10-07,2026-10-08,2026-10-09
0,arrival,ARN,362,351,349,353,355
1,arrival,BMA,20,23,23,22,21
2,arrival,GOT,80,69,77,82,74
3,arrival,KRN,2,2,2,2,2
4,arrival,LLA,15,13,15,15,16
5,arrival,MMX,23,21,22,23,22
6,arrival,OSD,6,6,6,6,5
7,arrival,RNB,6,6,6,6,4
8,arrival,UME,15,14,15,14,15
9,arrival,VBY,13,12,12,13,13


## 2) Identity. What IDENTIFIES one flight?

**Assumptions:** The flight leg identifier, i.e `flight_id` + `departure_date_utc` + `from_iata` + `to_iata` identifies exactly one flight per direction.

`legs` is used as a temporary `view` that should give both directions the same column names. Basically a preview of what the `dbt staging layer` will do later.

---

**Rows with UNIQUE flight identities:**
| direction | n_rows | n_flight_legs | n_duplicates | 
|:--|:--|:--|:--|
| arrival | 2649 | 2647 | 2 | 
| departure | 2635 | 2635 | 0 | 


---

**QUALIFY filtering:**
|direction |flight_id | departure_date_utc | from_iata | to_iata | status | scheduled_utc | actual_utc |
|:--|:--|:--|:--|:--|:--|:--|:--|
| arrival | BLX496	| 2026-10-07 02:00:00+02:00 |	RHO |	GOT |	LAN |	2026-10-08 00:50:00+02:00 |	2026-10-08 00:55:00+02:00 |
| arrival | BLX496	| 2026-10-07 02:00:00+02:00 |	RHO |	GOT | 	DEL |	2026-10-08 02:45:00+02:00 |	NaT |
| arrival |	D84528	| 2026-10-06 02:00:00+02:00 |	RAK |	ARN |	DEL |	2026-10-07 01:30:00+02:00 |	NaT |
| arrival |	D84528	| 2026-10-06 02:00:00+02:00 |	RAK |	ARN |	LAN | 	2026-10-07 02:10:00+02:00 |	2026-10-07 02:27:18+02:00 |


---

**Rows count but that Swedavia marked with DEL for deletion:**
| direction | n_rows | n_flight_legs | n_duplicates |
|:--|:--|:--|:--|
| arrival | 2206 | 2206 | 0 |
| departure | 2188 | 2188 | 0 |

In [4]:
# användandet av TEMP VIEW(view table), lever endast in memory, det skriver ingenting så read_only fungerar även här
# Samma column namn för båda riktningarna(arrival + departure) precis som staging layer i dbt ska göra.
con.sql("""
    CREATE OR REPLACE TEMP VIEW legs AS
    SELECT 'arrival' AS direction,
           flight_leg_identifier__flight_id AS flight_id,
           flight_leg_identifier__flight_departure_date_utc AS departure_date_utc,
           flight_leg_identifier__departure_airport_iata AS from_iata,
           flight_leg_identifier__arrival_airport_iata AS to_iata,
           location_and_status__flight_leg_status AS status,
           arrival_time__scheduled_utc AS scheduled_utc,
           arrival_time__actual_utc AS actual_utc
    FROM swedavia_raw.arrivals
    UNION ALL
    SELECT 'departure',
        flight_leg_identifier__flight_id,
        flight_leg_identifier__flight_departure_date_utc,
        flight_leg_identifier__departure_airport_iata,
        flight_leg_identifier__arrival_airport_iata,
        location_and_status__flight_leg_status,
        departure_time__scheduled_utc,
        departure_time__actual_utc
    FROM swedavia_raw.departures
""")

In [5]:
# Rader mot UNIKA flygidentiteter. Skillnaden är antalet duplicates
con.sql("""
    SELECT direction,
            COUNT(*) AS n_rows,
            COUNT(DISTINCT (flight_id, departure_date_utc, from_iata, to_iata)) AS n_flight_legs,
           n_rows - n_flight_legs AS n_duplicates
    FROM legs
    GROUP BY direction
    ORDER BY direction
""").df()

,direction,n_rows,n_flight_legs,n_duplicates
0,arrival,2649,2647,2
1,departure,2635,2635,0


In [6]:
# QUALIFY ska filtrera på en window function. Likt WHERE fast efter OVER()
# Ska finnas i Snowflake så mönstret ska fungera i dbt sen
# Koll för hur struligt det kommer bli med candidate keys

con.sql("""
    SELECT *
    FROM legs
    QUALIFY COUNT(*) OVER (
        PARTITION BY direction, flight_id, departure_date_utc, from_iata, to_iata
    ) > 1
    ORDER BY flight_id, scheduled_utc
""").df()


,direction,flight_id,departure_date_utc,from_iata,to_iata,status,scheduled_utc,actual_utc
0,arrival,BLX496,2026-10-07 02:00:00+02:00,RHO,GOT,LAN,2026-10-08 00:50:00+02:00,2026-10-08 00:55:00+02:00
1,arrival,BLX496,2026-10-07 02:00:00+02:00,RHO,GOT,DEL,2026-10-08 02:45:00+02:00,NaT
2,arrival,D84528,2026-10-06 02:00:00+02:00,RAK,ARN,DEL,2026-10-07 01:30:00+02:00,NaT
3,arrival,D84528,2026-10-06 02:00:00+02:00,RAK,ARN,LAN,2026-10-07 02:10:00+02:00,2026-10-07 02:27:18+02:00


In [7]:
# Samma räkning men utan rader som Swedavia marketat som DEL(borttagna)
con.sql("""
    SELECT direction,
           COUNT(*) AS n_rows,
           COUNT(DISTINCT (flight_id, departure_date_utc, from_iata, to_iata)) AS n_flight_legs,
           n_rows - n_flight_legs AS n_duplicates
    FROM legs
    WHERE status <> 'DEL'
    GROUP BY direction
    ORDER BY direction
""").df()

,direction,n_rows,n_flight_legs,n_duplicates
0,arrival,2206,2206,0
1,departure,2188,2188,0


### Findings related to question 2. identity

- With `DEL` rows: 
  - 2 duplicates were found, but only in `arrivals` table. 
- Without `DEL`: 
  - 0 duplicates were found in both directions.

- `DEL` = *Deleted* (Which points to a record removed by Swedavia) and *not* `CAN` = *Cancelled*. This is *two* different codes in the API spec.

- Both of the duplicates shows that Swedavia *deleted* the row and created a new one instead of updating it.


**What this means for the model:**
- This should indicate that the model will be notice when something has changed and will work if we were to add a `unique`-test on the `staging`-keys area after we're done filtering. IF **Swedavia** decides to change a time directly on an existing row one day instead of deleting it and creating a new one we would technically end up with *two* active rows sharing the *same* key.

- That would mean that the test would fail before the figure reaches the `serving layer dashboard`. Thinking in `W-A-P` terms, this would be the `A`(Audit) step. Audit and then Publish.

**The important parts for the modeling would then be these three parts:**

- 1) **Grain:** one row = one flight leg at a Swedavia airport, per direction.
- 2) **Key:** `direction` + `flight_id` + `departure_date_utc` + `from_iata` + `to_iata`, after removing `DEL`.
- 3) **Guard:** dbt `unique` test on the key in staging.

## 3) Which statuses count?

- Status distribution per direction(`arrivals`, `departures`) and how many of them actually has a time?


| direction | status | n_rows | percent | n_with_actual_time |
|:--|:--|:--|:--|:--|
| arrival | LAN | 1928 | 72.8 | 1928 |
| arrival | DEL | 443 | 16.7 | 0 |
| arrival | SCH | 256 | 9.7 | 0 |
| arrival | CAN | 22 | 0.8 | 0 |
| departure | ACT | 1926 | 73.1 | 1926 |
| departure | DEL | 447 | 17.0 | 1 |
| departure | SCH | 176 | 6.7 | 0 |
| departure | SEQ | 59 | 2.2 | 0 |
| departure | CAN | 26 | 1.0 | 1 |
| departure | FLS | 1 | 0.0 | 1 |

In [8]:
# Fördelning på statusar per riktning (arrivals och departures). Hur många flyg har en riktig tid?

con.sql("""
    SELECT direction,
            status,
            COUNT(*) AS n_rows,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY direction), 1) AS percent,
            COUNT(actual_utc) AS n_with_actual_time
    FROM legs
    GROUP BY direction, status
    ORDER BY direction, n_rows DESC
""").df()

,direction,status,n_rows,percent,n_with_actual_time
0,arrival,LAN,1928,72.8,1928
1,arrival,DEL,443,16.7,0
2,arrival,SCH,256,9.7,0
3,arrival,CAN,22,0.8,0
4,departure,ACT,1926,73.1,1926
5,departure,DEL,447,17.0,1
6,departure,SCH,176,6.7,0
7,departure,SEQ,59,2.2,0
8,departure,CAN,26,1.0,1
9,departure,FLS,1,0.0,1


In [9]:
# SCH status - Scheduled 
# SEQ status - Sequenced
# CAN status - Cancelled 
con.sql("""
    SELECT direction,
           status,
           CAST(scheduled_utc AT TIME ZONE 'Europe/Stockholm' AS DATE) AS local_date,
           COUNT(*) AS n_rows
    FROM legs
    WHERE status IN ('SCH', 'SEQ', 'CAN')
    GROUP BY ALL
    ORDER BY direction, status, local_date
""").df()

,direction,status,local_date,n_rows
0,arrival,CAN,2026-10-05,5
1,arrival,CAN,2026-10-06,2
2,arrival,CAN,2026-10-07,4
3,arrival,CAN,2026-10-08,6
4,arrival,CAN,2026-10-09,5
5,arrival,SCH,2026-10-08,1
6,arrival,SCH,2026-10-09,255
7,departure,CAN,2026-10-05,5
8,departure,CAN,2026-10-06,3
9,departure,CAN,2026-10-07,5


## 4) Delays. Which time field, and what happens around 900 seconds?

- Both directions have `scheduled_utc`, `estimated_utc` and `actual_utc`
- A delay should then be considered something like this:
    - Delay = `actual_utc` - `scheduled_utc`, only for completed flights (`LAN` / `ACT`).
    - Delayed = more than 15 minutes: `delay_seconds > 900`.

- Will keep a hard rule as per aviation standards which points towards a delay occurs if its over **15 minutes**. Therefore, I will consider a flight being delayed after a `900 second threshold`

In [10]:
# Förseningar i SEKUNDER. Faktiskt tid MINUS planerad tid. BARA för genomförda flyg (LAN(landed) och ACT(active))
# epoch() ger sekunderna enligt UNIX time. Använder detta för att få fram EXAKTA sekunder.
# Använder mig av CTE(Common table Expressoin) istället för messy subqueries.
con.sql("""
    WITH delays AS (
        SELECT direction,
               epoch(actual_utc) - epoch(scheduled_utc) AS delay_seconds
        FROM legs
        WHERE status IN ('LAN', 'ACT')
    )
    SELECT direction,
           COUNT(*) AS n_flights,
           COUNT(*) FILTER (WHERE delay_seconds < 0) AS n_early,
           MEDIAN(delay_seconds) AS median_s,
           QUANTILE_CONT(delay_seconds, 0.9) AS p90_s,
           MAX(delay_seconds) AS max_s,
           COUNT(*) FILTER (WHERE delay_seconds > 900) AS n_delayed,
           ROUND(100.0 * n_delayed / n_flights, 1) AS percent_delayed
    FROM delays
    GROUP BY direction
    ORDER BY direction 
""").df()

,direction,n_flights,n_early,median_s,p90_s,max_s,n_delayed,percent_delayed
0,arrival,1928,1341,-420.0,969.2,24244.0,201,10.4
1,departure,1926,931,9.5,1304.5,11605.0,276,14.3


In [11]:
# Är MINUS på median_s rimligt? Det måste innebära för tidiga departures och för tidiga arrivals.
# Kontroll för rimlighet. De tidigaste ankomsterna och vilka flyg det är.
con.sql("""
    SELECT flight_id, from_iata, to_iata, scheduled_utc, actual_utc,
           epoch(actual_utc) - epoch(scheduled_utc) AS delay_seconds
    FROM legs
    WHERE direction = 'arrival' AND status = 'LAN'
    ORDER BY delay_seconds
    LIMIT 5
""").df()

,flight_id,from_iata,to_iata,scheduled_utc,actual_utc,delay_seconds
0,DL204,JFK,ARN,2026-10-06 08:20:00+02:00,2026-10-06 07:21:49+02:00,-3491.0
1,SK904,EWR,ARN,2026-10-06 07:15:00+02:00,2026-10-06 06:17:52+02:00,-3428.0
2,NH221,HND,ARN,2026-10-06 07:15:00+02:00,2026-10-06 06:20:26+02:00,-3274.0
3,SK904,EWR,ARN,2026-10-05 07:15:00+02:00,2026-10-05 06:21:55+02:00,-3185.0
4,CA911,PEK,ARN,2026-10-05 17:20:00+02:00,2026-10-05 16:30:46+02:00,-2954.0


In [12]:
# Användningen av hårda gränsen på 900 sek(15 minuter). 3 olika vägar att gå för att räkna ut försening.
# Även här använder jag CTE istället gör messy subqueries.
con.sql("""
    WITH delays AS (
            SELECT direction,
                   epoch(actual_utc) - epoch(scheduled_utc) AS delay_seconds,
                   date_diff('minute', scheduled_utc, actual_utc) AS delay_minutes_diff
            FROM legs
            WHERE status IN ('LAN', 'ACT')
    )
    SELECT direction,
           COUNT(*) FILTER (WHERE delay_seconds > 900)     AS greater_than_900_s,
           COUNT(*) FILTER (WHERE delay_seconds >= 900)    AS greater_or_equal_900_s,
           COUNT(*) FILTER (WHERE delay_minutes_diff > 15) AS diff_greater_than_15_min
    FROM delays
    GROUP BY direction
    ORDER BY direction
""").df()

,direction,greater_than_900_s,greater_or_equal_900_s,diff_greater_than_15_min
0,arrival,201,206,196
1,departure,276,277,271


## 5) Codeshare. Flights with several flight numbers

- Same airplane under several different flight numbers.
- Are these flights calculated one time or more?

In [13]:
# Codeshare - Hur många flyg använder flera flight numbers i datan.
# Viktigt då dlt skapar flera child tables till sin parent table.
# Även här använder jag CTE som visats i klassrummet.
for table in ["arrivals", "departures"]:
    print(table)
    display(con.sql(f"""
        WITH cs_per_flight AS (
            SELECT f._dlt_id,
                   COUNT(c.value) AS n_codeshare
            FROM swedavia_raw.{table} AS f
            LEFT JOIN swedavia_raw.{table}__code_share_data AS c
                ON c._dlt_parent_id = f._dlt_id
            WHERE f.location_and_status__flight_leg_status <> 'DEL'
            GROUP BY f._dlt_id
        )
        SELECT COUNT(*) AS n_flights,
               COUNT(*) FILTER (WHERE n_codeshare > 0) AS n_with_codeshare,
               SUM(n_codeshare) AS n_codeshare_numbers,
               MAX(n_codeshare) AS max_per_flight
        FROM cs_per_flight
    """).df())

arrivals


,n_flights,n_with_codeshare,n_codeshare_numbers,max_per_flight
0,2206,890,2181.0,8


departures


,n_flights,n_with_codeshare,n_codeshare_numbers,max_per_flight
0,2188,866,2166.0,9


In [14]:
for table in ["arrivals", "departures"]:
    n = con.sql(f"""
        SELECT COUNT(*)
        FROM swedavia_raw.{table} AS f
        JOIN swedavia_raw.{table}__code_share_data AS c
            ON c._dlt_parent_id = f._dlt_id
        JOIN swedavia_raw.{table} AS own
            ON own.flight_leg_identifier__flight_id = c.value
           AND own.flight_leg_identifier__flight_departure_date_utc = f.flight_leg_identifier__flight_departure_date_utc
           AND own.flight_leg_identifier__departure_airport_iata = f.flight_leg_identifier__departure_airport_iata
           AND own.flight_leg_identifier__arrival_airport_iata = f.flight_leg_identifier__arrival_airport_iata
        WHERE f.location_and_status__flight_leg_status <> 'DEL'
    """).fetchone()[0]
    print(table, n)

arrivals 0
departures 0


## 6) Domestic flights in both arrivals and departure tables.
- Domestic flights between two Swedavia airports. Does the same flight exist in both tables?
- Does the departure on an airport and an arrival at a different one share the same key?
- Are there delays in both directions for the *same* flight? (Looking after correlation between late departure and late arrival or arrival/departure)

In [15]:
# Inhemska arrivals och departures. Delar flygen samma nyckel?

con.sql("""
    WITH departure AS (SELECT * FROM legs WHERE direction = 'departure' AND status <> 'DEL'),
         arrival AS (SELECT * FROM legs WHERE direction = 'arrival'   AND status <> 'DEL')
    SELECT (SELECT COUNT(*) FROM departure) AS n_departures,
           (SELECT COUNT(*) FROM arrival) AS n_arrivals,
           COUNT(*) AS n_in_both
    FROM departure
    JOIN arrival USING (flight_id, departure_date_utc, from_iata, to_iata)
""").df()

,n_departures,n_arrivals,n_in_both
0,2188,2206,631


In [16]:
# Samma flyg men TVÅ olika händelser.
# Innebär en försening vid avgång en försenad ankomst? 
con.sql("""
    WITH departure AS (SELECT *, epoch(actual_utc) - epoch(scheduled_utc) AS delay_seconds
                 FROM legs WHERE direction = 'departure' AND status = 'ACT'),
         arrival AS (SELECT *, epoch(actual_utc) - epoch(scheduled_utc) AS delay_seconds
                 FROM legs WHERE direction = 'arrival' AND status = 'LAN')
    SELECT COUNT(*) AS n_flights,
           COUNT(*) FILTER (WHERE departure.delay_seconds > 900) AS delayed_at_departure,
           COUNT(*) FILTER (WHERE arrival.delay_seconds > 900) AS delayed_at_arrival,
           COUNT(*) FILTER (WHERE departure.delay_seconds > 900 AND arrival.delay_seconds <= 900) AS caught_up_in_air
    FROM departure
    JOIN arrival USING (flight_id, departure_date_utc, from_iata, to_iata)
""").df()

,n_flights,delayed_at_departure,delayed_at_arrival,caught_up_in_air
0,541,40,36,11


## 7) Dates and which day does or should the fact carry?

- A flight can have times that could give a date and they dont always match eachother.
    - Swedish day for the planned time on *our* airport.
    - UTC-day for the same planned time. Since Sweden is +2 hours before UTC at the moment(summer time) that indicates that everything planned between 00:00 and 01:59 in swedish time gets placed in the previous day when counting in UTC.
- `departure_date_utc` is the date in the `flight key`. This is the day the flight `departed` from its *first* airport, in UTC. 
    - For an `arrival` from New York, this might be the previous day. 
    
- The *Swedish* day should be the one corresponding to the actual time. A flight scheduled for 23:55 that lands at 00:40 should therefore be assigned to the *following day*.

In [17]:
# Vilket datum är rätt. Jämförs mot Svensk dag för planerad tid.
# CTE istället för subquery som ovan.
con.sql("""
    WITH dates AS (
        SELECT direction,
               CAST(scheduled_utc AT TIME ZONE 'Europe/Stockholm' AS DATE) AS swedish_date,
               CAST(scheduled_utc AT TIME ZONE 'UTC' AS DATE) AS utc_date,
               CAST(departure_date_utc AT TIME ZONE 'UTC' AS DATE) AS leg_date,
               CAST(actual_utc AT TIME ZONE 'Europe/Stockholm' AS DATE) AS actual_swedish_date
        FROM legs
        WHERE status <> 'DEL'
    )
    SELECT direction,
           COUNT(*) AS n_flights,
           COUNT(*) FILTER (WHERE utc_date <> swedish_date) AS utc_differs,
           COUNT(*) FILTER (WHERE leg_date <> swedish_date) AS leg_date_differs,
           COUNT(*) FILTER (WHERE actual_swedish_date <> swedish_date) AS actual_differs
    FROM dates
    GROUP BY direction
    ORDER BY direction
""").df()

,direction,n_flights,utc_differs,leg_date_differs,actual_differs
0,arrival,2206,61,92,23
1,departure,2188,4,4,3
